# MAR sensitivity experiment
Focused validation for class/feature-dependent missingness.

In [ ]:

# ============================================================
# MAR sensitivity experiment for compound corruption benchmark
# Runs a focused missingness validation, not the full 64-grid.
# Outputs: raw_results_mar_sensitivity.csv
# ============================================================

import sys, subprocess, warnings, time
from pathlib import Path
warnings.filterwarnings("ignore")

try:
    from pmlb import fetch_data
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pmlb"])
    from pmlb import fetch_data

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, matthews_corrcoef, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
try:
    from xgboost import XGBClassifier
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])
    from xgboost import XGBClassifier
try:
    from lightgbm import LGBMClassifier
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
    from lightgbm import LGBMClassifier

OUT_DIR = Path('/kaggle/working/mar_sensitivity')
OUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = OUT_DIR / 'raw_results_mar_sensitivity.csv'
FAILED_PATH = OUT_DIR / 'failed_runs_mar_sensitivity.csv'

DATASETS = [
    'adult','mushroom','allhypo','clean2','dna','hypothyroid','kr_vs_kp','magic',
    'nursery','optdigits','pendigits','phoneme','satimage','segmentation','spambase','splice',
    'tic_tac_toe','vehicle','vowel','waveform_40','mfeat_fourier','mfeat_karhunen',
    'mfeat_morphological','mfeat_zernike'
]
SEEDS = [0,1,2,3,4]
# Focused settings: enough to compare missing-only and missing-containing interactions.
SETTINGS = [
    dict(setting='clean', label_noise=0.0, missing_rate=0.0, imbalance='natural'),
    dict(setting='missing_only', label_noise=0.0, missing_rate=0.2, imbalance='natural'),
    dict(setting='noise_missing', label_noise=0.2, missing_rate=0.2, imbalance='natural'),
    dict(setting='missing_imbalance', label_noise=0.0, missing_rate=0.2, imbalance=0.8),
    dict(setting='all_three', label_noise=0.2, missing_rate=0.2, imbalance=0.8),
]

CACHE_DIRS = [Path('/kaggle/input/pmlb-cache-csv'), Path('/kaggle/input/pmlb_cache_csv')]

def load_dataset(name):
    for cdir in CACHE_DIRS:
        f = cdir / f'{name}.csv'
        if f.exists():
            df = pd.read_csv(f)
            target = 'target' if 'target' in df.columns else df.columns[-1]
            return df.drop(columns=[target]), df[target]
    df = fetch_data(name)
    target = 'target' if 'target' in df.columns else df.columns[-1]
    return df.drop(columns=[target]), df[target]

def apply_label_noise(y, rate, seed):
    rng = np.random.default_rng(seed)
    y = pd.Series(y).reset_index(drop=True)
    arr = y.to_numpy().copy()
    if rate <= 0: return pd.Series(arr)
    classes = np.unique(arr)
    idx = rng.choice(len(arr), size=int(round(rate*len(arr))), replace=False)
    for i in idx:
        others = classes[classes != arr[i]]
        arr[i] = rng.choice(others)
    return pd.Series(arr)

def apply_imbalance(X, y, target_majority, seed):
    if target_majority == 'natural':
        return X.reset_index(drop=True), pd.Series(y).reset_index(drop=True)
    rng = np.random.default_rng(seed)
    X = X.reset_index(drop=True); y = pd.Series(y).reset_index(drop=True)
    counts = y.value_counts(); maj = counts.idxmax()
    maj_idx = y[y==maj].index.to_numpy()
    nonmaj_classes = [c for c in counts.index if c != maj]
    desired_nonmaj = int(round(len(maj_idx)*(1-target_majority)/target_majority))
    desired_nonmaj = max(desired_nonmaj, len(nonmaj_classes))
    selected = []
    for c in nonmaj_classes:
        ci = y[y==c].index.to_numpy()
        if len(ci): selected.append(rng.choice(ci))
    remaining = desired_nonmaj - len(selected)
    if remaining > 0:
        avail = np.array([i for i in y[y!=maj].index.to_numpy() if i not in set(selected)])
        if len(avail): selected += rng.choice(avail, size=min(remaining,len(avail)), replace=False).tolist()
    keep = np.concatenate([maj_idx, np.array(selected, dtype=int)])
    rng.shuffle(keep)
    return X.iloc[keep].reset_index(drop=True), y.iloc[keep].reset_index(drop=True)

def apply_mar_missingness(X, y, rate, seed):
    """Class/feature-dependent MAR-style missingness.
    Higher missingness probability is assigned to the minority class and to high values of each feature.
    The target itself is not used for imputation or model fitting; it only defines the corruption process.
    """
    if rate <= 0: return X.copy()
    rng = np.random.default_rng(seed)
    Xn = pd.DataFrame(X).copy().astype(float).reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)
    counts = y.value_counts()
    minority_class = counts.idxmin()
    class_factor = np.where(y.to_numpy()==minority_class, 1.6, 0.7)
    for col in Xn.columns:
        vals = Xn[col].to_numpy(dtype=float)
        med = np.nanmedian(vals)
        high_factor = np.where(vals > med, 1.3, 0.8)
        prob = rate * class_factor * high_factor
        prob = np.clip(prob, 0, min(0.85, rate*3.0))
        mask = rng.random(len(Xn)) < prob
        Xn.loc[mask, col] = np.nan
    return Xn

def get_models(seed, n_classes):
    models = {
        'LogisticRegression': Pipeline([('imp',SimpleImputer()),('sc',StandardScaler()),('clf',LogisticRegression(max_iter=1000, random_state=seed))]),
        'LinearSVM': Pipeline([('imp',SimpleImputer()),('sc',StandardScaler()),('clf',LinearSVC(max_iter=5000, random_state=seed))]),
        'KNN': Pipeline([('imp',SimpleImputer()),('sc',StandardScaler()),('clf',KNeighborsClassifier(n_neighbors=5))]),
        'DecisionTree': Pipeline([('imp',SimpleImputer()),('clf',DecisionTreeClassifier(random_state=seed))]),
        'RandomForest': Pipeline([('imp',SimpleImputer()),('clf',RandomForestClassifier(n_estimators=120, random_state=seed, n_jobs=-1))]),
        'ExtraTrees': Pipeline([('imp',SimpleImputer()),('clf',ExtraTreesClassifier(n_estimators=120, random_state=seed, n_jobs=-1))]),
        'HistGradientBoosting': Pipeline([('imp',SimpleImputer()),('clf',HistGradientBoostingClassifier(max_iter=120, random_state=seed))]),
    }
    xgb_obj = 'binary:logistic' if n_classes == 2 else 'multi:softprob'
    models['XGBoost'] = Pipeline([('imp',SimpleImputer()),('clf',XGBClassifier(n_estimators=120, max_depth=6, learning_rate=0.1, objective=xgb_obj, eval_metric='logloss', random_state=seed, n_jobs=-1, verbosity=0))])
    lgb_obj = 'binary' if n_classes == 2 else 'multiclass'
    models['LightGBM'] = Pipeline([('imp',SimpleImputer()),('clf',LGBMClassifier(n_estimators=120, objective=lgb_obj, random_state=seed, n_jobs=-1, verbose=-1))])
    return models

def metrics(model, X, y):
    pred = model.predict(X)
    labels = np.unique(y)
    recs = recall_score(y, pred, average=None, labels=labels, zero_division=0)
    minority = pd.Series(y).value_counts().idxmin()
    minority_recall = recs[list(labels).index(minority)] if minority in labels else np.nan
    return dict(
        accuracy=accuracy_score(y,pred),
        balanced_accuracy=balanced_accuracy_score(y,pred),
        macro_f1=f1_score(y,pred,average='macro',zero_division=0),
        mcc=matthews_corrcoef(y,pred),
        minority_recall=minority_recall
    )

existing = pd.read_csv(RAW_PATH) if RAW_PATH.exists() else pd.DataFrame()
completed = set(existing['run_key']) if len(existing) else set()
rows = existing.to_dict('records') if len(existing) else []
failed = []

for dname in DATASETS:
    print('Dataset:', dname)
    try:
        X, y = load_dataset(dname)
        X = pd.DataFrame(X).reset_index(drop=True)
        y = pd.Series(y).astype('category').cat.codes.reset_index(drop=True)
        if len(X) > 20000:
            _, X, _, y = train_test_split(X, y, test_size=20000, stratify=y, random_state=42)
            X = X.reset_index(drop=True); y = pd.Series(y).reset_index(drop=True)
    except Exception as e:
        failed.append(dict(dataset=dname, stage='load', error=str(e)[:300])); pd.DataFrame(failed).to_csv(FAILED_PATH,index=False); continue
    for seed in SEEDS:
        try:
            Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=seed)
        except Exception as e:
            failed.append(dict(dataset=dname, seed=seed, stage='split', error=str(e)[:300])); pd.DataFrame(failed).to_csv(FAILED_PATH,index=False); continue
        for sid, st in enumerate(SETTINGS):
            y_noise = apply_label_noise(ytr, st['label_noise'], 10000+seed*100+sid)
            X_tmp = Xtr.reset_index(drop=True)
            X_cor, y_cor = apply_imbalance(X_tmp, y_noise, st['imbalance'], 20000+seed*100+sid)
            X_cor = apply_mar_missingness(X_cor, y_cor, st['missing_rate'], 30000+seed*100+sid)
            X_eval = apply_mar_missingness(Xte.reset_index(drop=True), yte.reset_index(drop=True), st['missing_rate'], 40000+seed*100+sid)
            for mname, model in get_models(seed, pd.Series(y).nunique()).items():
                key = f'{dname}__seed{seed}__{st["setting"]}__{mname}'
                if key in completed: continue
                t0=time.time()
                try:
                    model.fit(X_cor, y_cor)
                    r = dict(run_key=key, dataset_name=dname, seed=seed, setting=st['setting'], model_name=mname,
                             label_noise=st['label_noise'], missing_rate=st['missing_rate'], imbalance_level=st['imbalance'], missingness_mechanism='MAR_class_feature', fit_eval_seconds=time.time()-t0)
                    r.update(metrics(model, X_eval, yte.reset_index(drop=True)))
                    rows.append(r); completed.add(key)
                    pd.DataFrame(rows).to_csv(RAW_PATH,index=False)
                except Exception as e:
                    failed.append(dict(dataset=dname, seed=seed, setting=st['setting'], model=mname, stage='fit_eval', error=str(e)[:300]))
                    pd.DataFrame(failed).to_csv(FAILED_PATH,index=False)
print('Done.', len(rows), 'rows')
